In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, pandas as pd

BASE_DIR = '/content/drive/MyDrive/ai4trade'
RAW_DIR  = f'{BASE_DIR}/data/raw'   # your folder with the csv.zip and where we’ll write parquet
os.makedirs(RAW_DIR, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("Found:")
for p in sorted(glob.glob(f'{RAW_DIR}/*.csv.zip')):
    print("  ", os.path.basename(p))


Mounted at /content/drive
RAW_DIR: /content/drive/MyDrive/ai4trade/data/raw
Found:
   trade_s_chn_m_hs_2023.csv.zip
   trade_s_chn_m_hs_2024.csv.zip
   trade_s_chn_m_hs_2025.csv.zip
   trade_s_usa_state_m_hs_2023.csv.zip
   trade_s_usa_state_m_hs_2024.csv.zip
   trade_s_usa_state_m_hs_2025.csv.zip


In [2]:
# EXACT filenames as per your screenshot, mapped to desired Parquet names
mapping = [
    ("trade_s_chn_m_hs_2023.csv.zip",       "CHN_2023.parquet"),
    ("trade_s_chn_m_hs_2024.csv.zip",       "CHN_2024.parquet"),
    ("trade_s_chn_m_hs_2025.csv.zip",       "CHN_2025.parquet"),
    ("trade_s_usa_state_m_hs_2023.csv.zip", "USA_2023.parquet"),
    ("trade_s_usa_state_m_hs_2024.csv.zip", "USA_2024.parquet"),
    ("trade_s_usa_state_m_hs_2025.csv.zip", "USA_2025.parquet"),
]

# Expand to full paths and validate that inputs exist
work = []
missing = []
for in_name, out_name in mapping:
    in_path  = os.path.join(RAW_DIR, in_name)
    out_path = os.path.join(RAW_DIR, out_name)
    if not os.path.exists(in_path):
        missing.append(in_name)
    work.append((in_path, out_path, in_name, out_name))

if missing:
    raise FileNotFoundError(f"These raw files were not found in {RAW_DIR}:\n" + "\n".join(missing))

import pandas as pd
pd.DataFrame(work, columns=["in_path","out_path","in_name","out_name"])


,in_path,out_path,in_name,out_name
0,/content/drive/MyDrive/ai4trade/data/raw/trade...,/content/drive/MyDrive/ai4trade/data/raw/CHN_2...,trade_s_chn_m_hs_2023.csv.zip,CHN_2023.parquet
1,/content/drive/MyDrive/ai4trade/data/raw/trade...,/content/drive/MyDrive/ai4trade/data/raw/CHN_2...,trade_s_chn_m_hs_2024.csv.zip,CHN_2024.parquet
2,/content/drive/MyDrive/ai4trade/data/raw/trade...,/content/drive/MyDrive/ai4trade/data/raw/CHN_2...,trade_s_chn_m_hs_2025.csv.zip,CHN_2025.parquet
3,/content/drive/MyDrive/ai4trade/data/raw/trade...,/content/drive/MyDrive/ai4trade/data/raw/USA_2...,trade_s_usa_state_m_hs_2023.csv.zip,USA_2023.parquet
4,/content/drive/MyDrive/ai4trade/data/raw/trade...,/content/drive/MyDrive/ai4trade/data/raw/USA_2...,trade_s_usa_state_m_hs_2024.csv.zip,USA_2024.parquet
5,/content/drive/MyDrive/ai4trade/data/raw/trade...,/content/drive/MyDrive/ai4trade/data/raw/USA_2...,trade_s_usa_state_m_hs_2025.csv.zip,USA_2025.parquet


In [5]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from zipfile import ZipFile
import io
import os

def convert_full_zipcsv_to_parquet(in_zip_csv: str, out_parquet: str, encoding="utf-8"):
    print(f"\n➡️ Loading {os.path.basename(in_zip_csv)} ...")

    # --- Step 1: inspect ZIP contents
    with ZipFile(in_zip_csv, 'r') as z:
        file_list = z.namelist()
        csv_files = [f for f in file_list if f.lower().endswith('.csv')]
        if not csv_files:
            raise ValueError(f"No CSV found in {in_zip_csv}. Files inside: {file_list}")
        if len(csv_files) > 1:
            print(f"⚠️ Multiple CSVs found, using the first one: {csv_files[0]}")
        csv_name = csv_files[0]

        # --- Step 2: read the CSV directly from memory
        with z.open(csv_name) as f:
            df = pd.read_csv(f, encoding=encoding, low_memory=False)

    print("Loaded shape:", df.shape)

    # --- Step 3: global deduplication
    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)
    print(f"Duplicates removed: {removed:,} | Final rows: {len(df):,}")

    # --- Step 4: write Parquet
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, out_parquet, compression="snappy")
    print(f"✅ Wrote: {out_parquet}")

    return before, len(df), df.shape[1]


In [6]:
results = []
for in_path, out_path, in_name, out_name in work:
    rows_in, rows_out, ncols = convert_full_zipcsv_to_parquet(in_path, out_path)
    results.append((in_name, out_name, rows_in, rows_out, ncols))

summary = pd.DataFrame(results, columns=["input_zip","output_parquet","rows_in","rows_out","n_cols"])
summary



➡️ Loading trade_s_chn_m_hs_2023.csv.zip ...
Loaded shape: (19122660, 13)
Duplicates removed: 908 | Final rows: 19,121,752
✅ Wrote: /content/drive/MyDrive/ai4trade/data/raw/CHN_2023.parquet

➡️ Loading trade_s_chn_m_hs_2024.csv.zip ...
Loaded shape: (19625529, 13)
Duplicates removed: 769 | Final rows: 19,624,760
✅ Wrote: /content/drive/MyDrive/ai4trade/data/raw/CHN_2024.parquet

➡️ Loading trade_s_chn_m_hs_2025.csv.zip ...
Loaded shape: (4971240, 13)
Duplicates removed: 54,758 | Final rows: 4,916,482
✅ Wrote: /content/drive/MyDrive/ai4trade/data/raw/CHN_2025.parquet

➡️ Loading trade_s_usa_state_m_hs_2023.csv.zip ...
Loaded shape: (22459534, 9)
Duplicates removed: 7,761 | Final rows: 22,451,773
✅ Wrote: /content/drive/MyDrive/ai4trade/data/raw/USA_2023.parquet

➡️ Loading trade_s_usa_state_m_hs_2024.csv.zip ...
Loaded shape: (22474549, 9)
Duplicates removed: 7,403 | Final rows: 22,467,146
✅ Wrote: /content/drive/MyDrive/ai4trade/data/raw/USA_2024.parquet

➡️ Loading trade_s_usa_state_

,input_zip,output_parquet,rows_in,rows_out,n_cols
0,trade_s_chn_m_hs_2023.csv.zip,CHN_2023.parquet,19122660,19121752,13
1,trade_s_chn_m_hs_2024.csv.zip,CHN_2024.parquet,19625529,19624760,13
2,trade_s_chn_m_hs_2025.csv.zip,CHN_2025.parquet,4971240,4916482,13
3,trade_s_usa_state_m_hs_2023.csv.zip,USA_2023.parquet,22459534,22451773,9
4,trade_s_usa_state_m_hs_2024.csv.zip,USA_2024.parquet,22474549,22467146,9
5,trade_s_usa_state_m_hs_2025.csv.zip,USA_2025.parquet,2036159,2036113,9
